# 1.0 Walking times between POIs

Estimated walking minutes from coordinates: haversine distance x 1.3 detour factor at 4.5 km/h, rounded up.
Legs longer than 40 minutes are outside a walking plan.

In [1]:
import pandas as pd

from corga import DATASET_PATH
from corga.data.store import POIStore
from corga.planning.geo import haversine_km
from corga.planning.rules import PlanningRules

store = POIStore.from_json(DATASET_PATH)
pois = store.all()
rules = PlanningRules()
print(rules.walking)

Walking times are estimates: straight-line distance x 1.3 at 4.5 km/h, not a routed path.


## 1. Walking-minute matrix

In [2]:
minutes = pd.DataFrame(
    [[rules.walking.minutes(a.location, b.location) for b in pois] for a in pois],
    index=[p.id for p in pois],
    columns=[p.id for p in pois],
)
minutes.iloc[:12, :12]

,CF-001,CF-002,CF-003,CF-004,CF-005,CF-006,CF-007,CF-008,CF-009,CF-010,CF-011,CF-012
CF-001,0,18,12,10,9,7,9,16,10,8,7,9
CF-002,18,0,8,9,11,14,10,4,9,12,13,10
CF-003,12,8,0,3,4,10,5,6,5,5,6,5
CF-004,10,9,3,0,3,8,2,7,4,4,5,3
CF-005,9,11,4,3,0,8,4,9,5,2,3,4
CF-006,7,14,10,8,8,0,6,12,5,8,7,5
CF-007,9,10,5,2,4,6,0,8,2,4,4,1
CF-008,16,4,6,7,9,12,8,0,7,10,11,8
CF-009,10,9,5,4,5,5,2,7,0,6,6,2
CF-010,8,12,5,4,2,8,4,10,6,0,2,4


In [3]:
minutes.stack().describe().to_frame("minutes between two POIs").T

,count,mean,std,min,25%,50%,75%,max
minutes between two POIs,2500.0,50.2392,116.737008,0.0,6.0,10.0,27.0,536.0


## 2. Distance from the Old Town

In [4]:
old_fortress = store.get("CF-001")
far = (
    pd.DataFrame(
        {
            "name": p.name,
            "km": round(haversine_km(old_fortress.location, p.location), 2),
            "walk_min": rules.walking.minutes(old_fortress.location, p.location),
        }
        for p in pois
    )
    .set_axis([p.id for p in pois])
    .sort_values("walk_min", ascending=False)
)
far.head(8)

,name,km,walk_min
CF-047,Perdita's Glass Art,28.46,494
CF-046,Ilios Jewellery Workshop,23.51,408
CF-021,Achilleion Palace,7.03,122
CF-022,Mon Repos Museum of Palaiopolis,1.88,33
CF-023,Mon Repos Park,1.78,31
CF-024,Boiler Food Bar,1.78,31
CF-043,Avli Restaurant Corfu,1.46,26
CF-044,Maridaki Seafood Restaurant,1.45,26


In [5]:
nearest = minutes.where(minutes > 0).min(axis=1)
isolated = nearest[nearest > rules.max_leg_minutes]
pd.DataFrame({"name": [store.get(i).name for i in isolated.index], "nearest_poi_min": isolated})

,name,nearest_poi_min
CF-021,Achilleion Palace,91.0
CF-046,Ilios Jewellery Workshop,94.0
CF-047,Perdita's Glass Art,94.0


## 3. Five named places in a 90-minute window

In [6]:
named_ids = ["CF-001", "CF-002", "CF-021", "CF-022", "CF-017"]
sub = minutes.loc[named_ids, named_ids]
sub.index = sub.columns = [f"{i} {store.get(i).name[:18]}" for i in named_ids]
sub

,CF-001 Old Fortress,CF-002 New Fortress,CF-021 Achilleion Palace,CF-022 Mon Repos Museum o,CF-017 Museum of Asian Ar
CF-001 Old Fortress,0,18,122,33,8
CF-002 New Fortress,18,0,122,39,12
CF-021 Achilleion Palace,122,122,0,91,126
CF-022 Mon Repos Museum o,33,39,91,0,38
CF-017 Museum of Asian Ar,8,12,126,38,0


## 4. Old Town cluster

In [7]:
cluster = ["CF-001", "CF-003", "CF-005", "CF-006", "CF-007", "CF-009", "CF-012", "CF-017", "CF-029"]
minutes.loc[cluster, cluster]

,CF-001,CF-003,CF-005,CF-006,CF-007,CF-009,CF-012,CF-017,CF-029
CF-001,0,12,9,7,9,10,9,8,12
CF-003,12,0,4,10,5,5,5,5,2
CF-005,9,4,0,8,4,5,4,2,4
CF-006,7,10,8,0,6,5,5,8,11
CF-007,9,5,4,6,0,2,1,4,6
CF-009,10,5,5,5,2,0,2,6,7
CF-012,9,5,4,5,1,2,0,4,6
CF-017,8,5,2,8,4,6,4,0,5
CF-029,12,2,4,11,6,7,6,5,0
